# Oracle seed-11 completion + k=2 pruned-graph LSTM check

Runs the two GPU-blocked parts from the 2026-07-16 /crs session, both pre-registered in
`experiments/topology_ablation/preregistration_baseline_completion_and_k2.md`:

- **Part 2 — Oracle seed-11 restore.** Re-train `L_upQ_component0_seed11` (checkpoint lost in a
  drive merge) to fill the blank oracle log-NSE/KGE columns in `PAPER_TABLE.md`. Success:
  median test NSE reproduces 0.703 ± 0.005 (determinism).
- **Part 3 — k=2 graph check.** Re-train the oracle + realizable conditions on the
  hydrography-realistic **in-degree≤2** pruned graph (266 edges vs 624). Success: k=2 realizable
  Δ vs L within ±0.010 of the full-graph +0.027 → confirms the *LSTM* (not just the R1 proxy) is
  pruning-invariant, closing the over-connectivity threat at the model level.

**Idempotent — safe to Run All repeatedly.** Every training cell skips if its run already
completed (checks `test/model_epoch030/test_metrics.csv`); the feature-build cell skips if the
k=2 pickles already exist with a correctly-named index. Re-running after a partial failure only
does the missing work.

**Runtime → Change runtime type → T4 GPU** before running. Same clone/mount/symlink pattern as
`colab_multiseed.ipynb`.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (find CAMELS on Drive; set runs dir)

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # leave blank to auto-detect
SEED=11  # oracle restore + k2 check are seed-11 (matches the full-graph seed-11 deltas)
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('seed', SEED)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4 like the other notebooks)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Ensure the L seed-11 baseline exists (needed as the Δ-reference + for the predicted feature)

If your Drive runs already have `L_component0_seed11` and its `_Lfullspan_eval_seed11`, this is a
no-op. Otherwise it trains L and runs the full-span eval that the predicted-Q feature needs.

In [ ]:
%cd {REPO_DIR}
import glob
BASIN='topology_analysis/phase1_network_discovery/outputs/component0_basins.txt'
!python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -1
Ldir=f'{REPO_DIR}/runs/topology_ablation/component0/L_component0_seed{SEED}'
if not os.path.isfile(f'{Ldir}/test/model_epoch030/test_metrics.csv'):
    !python experiments/topology_ablation/make_configs.py --network component0 --basin-file {BASIN} --seed {SEED} --device cuda:0 --epochs 30
    !python neuralhydrology/nh_run.py train --config-file experiments/topology_ablation/configs/L_component0_seed{SEED}.yaml 2>&1 | tail -2
    ts=sorted(glob.glob(f'{Ldir}_*'))
    if ts: os.rename(ts[-1], Ldir)
    !python neuralhydrology/nh_run.py evaluate --run-dir {Ldir} --epoch 30 2>&1 | tail -1
print('L seed', SEED, 'ready:', os.path.isfile(f'{Ldir}/test/model_epoch030/test_metrics.csv'))

## Cell 8 — PART 2: Oracle seed-11 restore (full graph)

Rebuilds the observed-upstream-Q feature on the FULL graph and trains `L_upQ_component0_seed11`.
Determinism check prints at the end.

In [ ]:
%cd {REPO_DIR}
FEAT='experiments/topology_ablation/features'
B=f'{REPO_DIR}/runs/topology_ablation/component0'
oracle_metrics=f'{B}/L_upQ_component0_seed{SEED}/test/model_epoch030/test_metrics.csv'
# Idempotent: skip if this run already completed (test_metrics.csv is written only after evaluate).
if os.path.isfile(oracle_metrics):
    print(f'[skip] L_upQ_component0_seed{SEED} already complete')
else:
    # build observed upstream_q on the full graph (regenerates the pickle; gitignored so not in clone)
    !python experiments/topology_ablation/build_upstream_discharge_feature.py --network component0 --lag-days 1 2>&1 | tail -1
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_component0_lag1.p --cond-name L_upQ 2>&1 | tail -2
import pandas as pd
m=pd.read_csv(oracle_metrics)
med=m['NSE'].median()
print(f'\nPART 2 oracle seed{SEED} median NSE = {med:.4f}  (pre-reg: reproduce 0.703 +/- 0.005 -> {abs(med-0.703)<=0.005})')

## Cell 9 — PART 3: build k=2 pruned-graph features

The k=2 edge set `component0_edges_k2.csv` is committed. The k=2 feature pickles are gitignored
(heavy), so regenerate them here from the committed edge set + the seed-11 full-span predictions.

In [ ]:
%cd {REPO_DIR}
import pickle, numpy as np, pandas as pd, networkx as nx
from pathlib import Path
OUTD=Path('experiments/topology_ablation/features'); OUTD.mkdir(parents=True,exist_ok=True)
k2_obs=OUTD/'upstream_q_obs_component0_k2_lag1.p'
k2_pred=OUTD/'upstream_q_pred_component0_k2_lag1.p'
# Idempotent: only rebuild if a k2 feature is missing OR has an unnamed index (the old KeyError bug).
def _ok(p):
    if not p.exists(): return False
    d=pickle.load(open(p,'rb')); return d[next(iter(d))].index.name=='date'
if _ok(k2_obs) and _ok(k2_pred):
    print('[skip] k2 features already built with named index')
else:
    P1=Path('topology_analysis/phase1_network_discovery/outputs')
    BASE=Path('runs/topology_ablation/component0')
    TOPO=Path('datasets/camels_us/camels_attributes_v2.0/camels_topo.txt')
    pruned=pd.read_csv(P1/'component0_edges_k2.csv',dtype={'parent_id':str,'child_id':str})
    basins=[l.strip() for l in open(P1/'component0_basins.txt') if l.strip()]
    area=pd.read_csv(TOPO,sep=';',dtype={'gauge_id':str}).set_index('gauge_id')['area_gages2'].to_dict()
    fs=pickle.load(open(BASE/f'_Lfullspan_eval_seed{SEED}/test/model_epoch030/test_results.p','rb'))
    # Index MUST be named 'date' — NH concatenates additional_feature_files onto the per-basin
    # dynamic df (indexed by a 'date' column); an UNNAMED index -> KeyError: 'date' at training
    # (same root cause as commit 52bd535 in the predicted-Q builder).
    def _didx(d): return pd.DatetimeIndex(pd.to_datetime(d['1D']['xr']['date'].values), name='date')
    obs={b:pd.Series(d['1D']['xr']['QObs(mm/d)_obs'].values.squeeze(),index=_didx(d)) for b,d in fs.items()}
    pred={b:pd.Series(d['1D']['xr']['QObs(mm/d)_sim'].values.squeeze(),index=_didx(d)) for b,d in fs.items()}
    G=nx.DiGraph(); G.add_nodes_from(basins)
    for _,r in pruned.iterrows():
        if r['parent_id'] in basins and r['child_id'] in basins: G.add_edge(r['parent_id'],r['child_id'])
    def build(src,tag):
        feats={}
        for b in basins:
            if b not in src: continue
            idx=src[b].index  # named 'date' (inherited from _didx) -> concatenates cleanly in NH
            parents=list(G.predecessors(b))
            if not parents: feats[b]=pd.DataFrame({'upstream_q':np.zeros(len(idx))},index=idx); continue
            agg=pd.Series(0.0,index=idx); wsum=0.0
            for p in parents:
                if p not in src: continue
                pa=float(area.get(p,0.0)); agg=agg.add((src[p].reindex(idx)*pa).fillna(0.0),fill_value=0.0); wsum+=pa
            if wsum>0: agg=agg/wsum
            feats[b]=pd.DataFrame({'upstream_q':agg.shift(1).fillna(0.0).values},index=idx)
        out=OUTD/f'upstream_q_{tag}_component0_k2_lag1.p'; pickle.dump(feats,open(out,'wb'))
        assert feats[next(iter(feats))].index.name=='date', 'index must be named date'
        print('wrote',out.name,'(index name =',feats[next(iter(feats))].index.name,')')
    build(obs,'obs'); build(pred,'pred')

## Cell 10 — PART 3: train k=2 oracle + realizable, compare Δ vs L

In [ ]:
%cd {REPO_DIR}
FEAT='experiments/topology_ablation/features'
B=f'{REPO_DIR}/runs/topology_ablation/component0'
# Idempotent: skip either k2 condition if its run already completed.
def _done(cond): return os.path.isfile(f'{B}/{cond}_component0_seed{SEED}/test/model_epoch030/test_metrics.csv')
# k=2 oracle
if _done('L_upQ_k2'):
    print(f'[skip] L_upQ_k2_component0_seed{SEED} already complete')
else:
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_obs_component0_k2_lag1.p --cond-name L_upQ_k2 2>&1 | tail -2
# k=2 realizable
if _done('L_upQpred_k2'):
    print(f'[skip] L_upQpred_k2_component0_seed{SEED} already complete')
else:
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_pred_component0_k2_lag1.p --cond-name L_upQpred_k2 2>&1 | tail -2

## Cell 11 — Verdict: k=2 Δ vs full-graph Δ (paired per-basin, connected basins)

In [ ]:
%cd {REPO_DIR}
import pandas as pd
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def nse(cond):
    p=f'{B}/{cond}_component0_seed{SEED}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
L=nse('L'); ok2=nse('L_upQ_k2'); pk2=nse('L_upQpred_k2')
import pickle, numpy as np
feat=pickle.load(open('experiments/topology_ablation/features/upstream_q_pred_component0_k2_lag1.p','rb'))
conn=[b for b in feat if np.abs(feat[b]['upstream_q'].values).mean()>0]
def paired(x):
    c=[b for b in conn if b in x.index and b in L.index]
    return float((x.loc[c]-L.loc[c]).median())
print('=== PART 3 verdict (seed %d, connected basins) ===' % SEED)
if ok2 is not None: print(f'k2 oracle    Δ vs L = {paired(ok2):+.4f}   (full-graph oracle    +0.037)')
if pk2 is not None:
    d=paired(pk2); print(f'k2 realizable Δ vs L = {d:+.4f}   (full-graph realizable +0.027)')
    print(f'\nPRE-REG: k2 realizable within +/-0.010 of +0.027 -> {abs(d-0.027)<=0.010}  (>=+0.010 & positive -> {d>=0.010}) ')
    print('PASS = LSTM (not just R1 proxy) is pruning-invariant; over-connectivity threat closed at model level.')

## Done

Runs persist to Drive (`neural_hydrology_runs`). To bring results back into the repo for the
paper table, copy the new `L_upQ_component0_seed11`, `L_upQ_k2_component0_seed11`,
`L_upQpred_k2_component0_seed11` folders' `test/model_epoch030/` from Drive, then re-run
`build_paper_table.py` and `analyze_metric_honesty.py` locally to fill the oracle columns.